In [ ]:
from pathlib import Path
import time
import json
from tqdm.auto import tqdm
import gc

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import f1_score, roc_auc_score, average_precision_score, matthews_corrcoef, precision_recall_curve, precision_score, recall_score
from sklearn.model_selection import train_test_split
from sklearn.inspection import permutation_importance
import shutil
import sys
scripts_dir = (Path.cwd() / ".." / "scripts").resolve()
sys.path.append(str(scripts_dir))

from config import final_data_path, hf_token


In [2]:
!pip install -U tabpfn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 849.3/849.3 kB 16.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 595.9/595.9 kB 23.3 MB/s eta 0:00:00


Setting variables for data paths.

In [ ]:
final_data_path = Path(final_data_path)
features_dir = final_data_path
results_dir = final_data_path / "modeling_results" / "fm_tuned"
results_dir.mkdir(parents=True, exist_ok=True)

layer_files = [
    "layer_a_bts",
    "layer_b_bts_meteostat",
    "layer_c_bts_gdelt",
    "layer_d_bts_meteostat_gdelt",
]

target_col = "dep_del15"
non_feature_cols = ["flight_id", "year", target_col]

datasets = {}
for name in layer_files:
    key = f"{name}_subsampled"
    df = pd.read_parquet(features_dir / f"{key}_features.parquet")
    datasets[key] = df
    print(f"{key}: {df.shape}")

layer_a_bts_subsampled: (6966559, 28)
layer_b_bts_meteostat_subsampled: (6966559, 32)
layer_c_bts_gdelt_subsampled: (6966559, 43)
layer_d_bts_meteostat_gdelt_subsampled: (6966559, 47)


In [4]:
def get_train_test(key: str):
    """Split one dataset by year and drop non-feature columns from X."""
    df = datasets[key]
    train = df[df["year"] == 2024]
    test = df[df["year"] == 2025]

    feature_cols = [c for c in df.columns if c not in non_feature_cols]
    X_train, y_train = train[feature_cols], train[target_col]
    X_test, y_test = test[feature_cols], test[target_col]
    return X_train, y_train, X_test, y_test

In [ ]:
import os
os.environ["TABPFN_MODEL_VERSION"] = "v3.5"
os.environ["TABPFN_TOKEN"] = hf_token

In [ ]:
from tabpfn.finetuning import FinetunedTabPFNClassifier
from tabpfn.constants import ModelVersion

Creating a 100 000 row subsample for the parameter search using Layer D.

In [ ]:

search_sample_size = 100_000
ft_search_layers = ["layer_d_bts_meteostat_gdelt_subsampled"]

ft_search_samples = {}
for search_layer in ft_search_layers:
    X_train_full, y_train_full, _, _ = get_train_test(search_layer)
    ft_search_df = X_train_full.copy()
    ft_search_df[target_col] = y_train_full.values
    ft_search_sample = ft_search_df.groupby(target_col, group_keys=False).apply(
        lambda g: g.sample(frac=search_sample_size / len(ft_search_df), random_state=42)
    )

    X_search = ft_search_sample.drop(columns=[target_col])
    y_search = ft_search_sample[target_col]
    X_ft_train, X_ft_val, y_ft_train, y_ft_val = train_test_split(
        X_search, y_search, test_size=0.15, stratify=y_search, random_state=42
    )
    ft_search_samples[search_layer] = (X_ft_train, X_ft_val, y_ft_train, y_ft_val)
    print(f"{search_layer}: {len(X_search):,} rows sampled")

layer_d_bts_meteostat_gdelt_subsampled: 100,000 rows sampled


/tmp/ipykernel_24/3120486361.py:9: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  ft_search_sample = ft_search_df.groupby(target_col, group_keys=False).apply(


Running the search for the best learning rate and number of epochs for finetuning.

In [ ]:
class HistoryLogger:
    """Minimal logger conforming to tabpfn.finetuning.logging.FinetuningLogger --
    captures per-epoch metrics AND prints live, so progress is visible during the run
    instead of dumping all at once when fit() returns."""
    def __init__(self):
        self.epoch_records = []

    def setup(self, config): pass
    def log_step(self, metrics, step): pass
    def log_epoch(self, metrics, step):
        self.epoch_records.append({**metrics, "step": step})
        if val_metric_key in metrics:
            print(f"    epoch {int(metrics['train/epoch'])}: val_auc={metrics[val_metric_key]:.4f}", flush=True)
    def finish(self): pass


tabpfn_n_estimators = 2
tabpfn_subsample_samples = 20_000
learning_rates = [3e-5, 1e-5, 3e-6, 1e-6, 3e-7]
max_epochs = 30
patience = 8
val_metric_key = "val/ROC AUC"

ft_search_results = []
ft_checkpoint_dir = results_dir / "finetune_search_checkpoints"
ft_search_csv_path = results_dir / "finetune_lr_search_layer_d.csv"

for search_layer, (X_ft_train, X_ft_val, y_ft_train, y_ft_val) in ft_search_samples.items():
    for lr in learning_rates:
        print(f"\n=== {search_layer} | learning_rate={lr} ===")
        t0 = time.time()
        run_dir = ft_checkpoint_dir / search_layer / f"lr_{lr}"
        shutil.rmtree(run_dir, ignore_errors=True)  # start from the base model, never resume an old checkpoint
        clf = None

        try:
            history_logger = HistoryLogger()

            clf = FinetunedTabPFNClassifier(
                model_version=ModelVersion.V3_5,
                device="cuda",
                epochs=max_epochs,
                learning_rate=lr,
                early_stopping=True,
                early_stopping_patience=patience,
                eval_metric="roc_auc",
                experiment_logger=history_logger,
                n_estimators_finetune=tabpfn_n_estimators,
                n_estimators_validation=tabpfn_n_estimators,
                n_estimators_final_inference=tabpfn_n_estimators,
                n_inference_subsample_samples=tabpfn_subsample_samples,
                n_finetune_ctx_plus_query_samples=10_000,
                random_state=42,
            )
            clf.fit(
                X_ft_train, y_ft_train,
                X_val=X_ft_val, y_val=y_ft_val,
                output_dir=run_dir,
            )

            val_history = [r for r in history_logger.epoch_records if val_metric_key in r]
            best_record = max(val_history, key=lambda r: r[val_metric_key])
            result = {
                "best_epoch": int(best_record["train/epoch"]),
                "best_val_auc": best_record[val_metric_key],
                "epochs_run": len(val_history),
                "status": "ok",
            }

        except Exception as e:
            # OOM or any other failure on this LR: log it and continue with the next one
            print(f"  FAILED: {type(e).__name__}: {e}")
            result = {
                "best_epoch": None,
                "best_val_auc": None,
                "epochs_run": None,
                "status": f"failed: {type(e).__name__}",
            }

        finally:
            # save after every LR, so a crash later still leaves the earlier results on disk
            ft_search_results.append({
                "layer": search_layer,
                "learning_rate": lr,
                **result,
                "total_search_seconds": time.time() - t0,
            })
            pd.DataFrame(ft_search_results).to_csv(ft_search_csv_path, index=False)

            # free GPU memory and disk space (checkpoints aren't needed for the search)
            del clf
            gc.collect()
            torch.cuda.empty_cache()
            shutil.rmtree(run_dir, ignore_errors=True)

ft_search_df_results = pd.DataFrame(ft_search_results)
ft_search_df_results

/usr/local/lib/python3.12/dist-packages/tabpfn/finetuning/finetuned_base.py:877: UserWarning: Output dir present but no checkpoint file found for train_size=85000. Starting from default checkpoint and epoch 0
  get_checkpoint_path_and_epoch_from_output_dir(



=== layer_d_bts_meteostat_gdelt_subsampled | learning_rate=3e-05 ===


tabpfn-v3.5-20260909.safetensors:   0%|          | 0.00/876M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/36.0 [00:00<?, ?B/s]

Finetuning Epoch 1/30:   0%|          | 0/9 [00:00<?, ?it/s]

...

Finetuning Epoch 7/30:   0%|          | 0/9 [00:00<?, ?it/s]

    epoch 6: val_auc=0.6970


Finetuning Epoch 8/30:   0%|          | 0/9 [00:00<?, ?it/s]

    epoch 7: val_auc=0.6970


,layer,learning_rate,best_epoch,best_val_auc,epochs_run,status,total_search_seconds
0,layer_d_bts_meteostat_gdelt_subsampled,3.000000e-05,19,0.698584,28,ok,4089.666103
1,layer_d_bts_meteostat_gdelt_subsampled,1.000000e-05,0,0.696743,8,ok,1170.939305
2,layer_d_bts_meteostat_gdelt_subsampled,3.000000e-06,11,0.697409,20,ok,2908.641180
3,layer_d_bts_meteostat_gdelt_subsampled,1.000000e-06,12,0.697417,19,ok,2762.715104
4,layer_d_bts_meteostat_gdelt_subsampled,3.000000e-07,0,0.697047,8,ok,1171.226640
